In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.insert(0, os.path.abspath("../.."))

In [4]:
#  imports & setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import gc
import time

gc.enable()

import torch


from src.config import EXP_CONFIGS
from src.utils import set_seed, save_config




main_folder = "../.."
data_csv = os.path.join(main_folder, "data", "train.csv")
img_folder = os.path.join(main_folder, "data", "train")
out_dir = os.path.join(main_folder, "outputs", "exp7_latefusion_swin_mse_MLPHead")

df = pd.read_csv(data_csv)

TARGET = "Pawpularity"
tab_cols = [c for c in df.columns if c not in ["Id", TARGET]]

os.makedirs(out_dir, exist_ok=True)
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error

# paths
main_folder = "../.."
img_oof_path  = os.path.join(main_folder, "outputs","extra", "exp3_swin_384_strong_MSE_MLPHead", "oof_detail.csv")
gbdt_oof_path = os.path.join(main_folder, "outputs", "exp1", "oof_detail.csv")  

img_oof  = pd.read_csv(img_oof_path)   # columns: Id, fold, ytrue, oof_pred, abs_err
gbdt_oof = pd.read_csv(gbdt_oof_path)  # same columns



In [7]:
# check for alignment of Ids and folds
img_map = img_oof[["Id", "fold"]].sort_values("Id").reset_index(drop=True)
gbdt_map = gbdt_oof[["Id", "fold"]].sort_values("Id").reset_index(drop=True)

assert img_map.equals(gbdt_map), "Fold assignment differs! RMSE per fold will be wrong."


In [8]:
assert set(img_oof["Id"]) == set(gbdt_oof["Id"]), "IDs differ!"

In [9]:

# inner join on Id to  alignment
oof = img_oof.merge(
    gbdt_oof[["Id", "oof_pred"]].rename(columns={"oof_pred": "gbdt_oof"}),
    on="Id",
    how="inner",
)

y_true = oof["ytrue"].values
img_pred = oof["oof_pred"].values
gbdt_pred = oof["gbdt_oof"].values

In [10]:
img_rmse = root_mean_squared_error(y_true, img_pred)
gbdt_rmse = root_mean_squared_error(y_true, gbdt_pred)
print("Image OOF RMSE:", img_rmse)
print("GBDT OOF RMSE:", gbdt_rmse)

Image OOF RMSE: 17.73602090865089
GBDT OOF RMSE: 20.668644721159303


### simple average

In [14]:
from src.utils import late_fusion_from_oof

# oof
oof_fused, oof_rmse, info = late_fusion_from_oof(
    oof,
    img_col="oof_pred",
    tab_col="gbdt_oof",
    y_col="ytrue",
    fold_col="fold",
    mode="simple",      # or "simple" | "weighted", simple is just average 
)

print("Global OOF RMSE:", oof_rmse)
print("Weights:", info["weight_a"], info["weight_b"])
print("Fold-wise RMSE:", info["fold_rmse"])
print("Avg:", info["fold_rmse_mean"], "Std:", info["fold_rmse_std"])

oof_fused = oof_fused.rename(columns={"oof_pred": "swin_oof"})

# save new OOF 
oof_fused.to_csv(os.path.join(main_folder, "outputs", "exp7", "swin_GBDT_average_oof_detail.csv"),
                 index=False)



Global OOF RMSE: 18.355570783153382
Weights: 0.5 0.5
Fold-wise RMSE: {1: 18.588220709666576, 2: 18.510667684141037, 3: 17.945897199437855, 4: 18.269423579796264, 5: 18.456268338549616}
Avg: 18.35409550231827 Std: 0.22959265590963918


### weighted average

In [13]:
oof_fused, oof_rmse, info = late_fusion_from_oof(
    oof,
    img_col="oof_pred",
    tab_col="gbdt_oof",
    y_col="ytrue",
    fold_col="fold",
    mode="weighted",      # or "simple" | "weighted", simple is just average 
    n_grid=101,
)

print("Global OOF RMSE:", oof_rmse)
print("Weights:", info["weight_a"], info["weight_b"])
print("Fold-wise RMSE:", info["fold_rmse"])
print("Avg:", info["fold_rmse_mean"], "Std:", info["fold_rmse_std"])

oof_fused = oof_fused.rename(columns={"oof_pred": "swin_oof"})
oof_fused["abs_err"] = (oof_fused["ytrue"] - oof_fused["final_pred"]).abs()
# save new OOF 
oof_fused.to_csv(os.path.join(main_folder, "outputs", "exp7", "swin_GBDT_weighted_oof_detail.csv"),
                 index=False)
oof_fused.sort_values("abs_err", ascending=False).head(50).to_csv(
    os.path.join(main_folder, "outputs", "exp7",  "swin_GBDT_top50_errors.csv"), index=False
)

Global OOF RMSE: 17.708187933595763
Weights: 0.91 0.08999999999999997
Fold-wise RMSE: {1: 17.77905105196138, 2: 18.018943939690605, 3: 17.302801187689283, 4: 17.492950288494097, 5: 17.936707983880975}
Avg: 17.70609089034327 Std: 0.269983688368173
